# Documents Notebook

For this chatbot I will be using bash man pages, tealdeer output and the output from the bash help system for builtin commands.

There will need to be 2 pipelines.
- The first will be the data ingestion pipeline, which is in this notebook.
- Then we will need a query pipeline that will use our domain specific embeddings as the context for an LLM prompt. This is the RAG implementation that will be built. It is located in bashbot.ipynb and the associated script.

The dependencies are in requirements.txt.

For the default Ollama implementation, you will need to install Ollama and pull the following models:

- _nomic-embed-text_
- _llama3_
- _mistral-small3.2_
- _granite3.3:8b_


### Data Generation
- Generate man pages for all known commands (from bash)

    `compgen -c | sort | uniq -d | xargs -I {} sh -c "man '{}' > '{}.man'" 2>>noman.txt`

- Delete empty files (no man page)

    `find . -type f -empty -delete`

- Use tealdeer for additional data

    `tldr --update`

- Generate tldr files

    `compgen -c | sort | uniq -d | xargs -I {} sh -c "tldr '{}' > '{}.tldr'" 2>>notldr.txt`

- Delete empty files (no tldr entry)

    `find . -type f -empty -delete`

- Replace {COMMAND} with <CMD> to avoid LLM errors with reserved word structure.

    `find ./ -type f -name "*.txt" -exec sed -i -e 's/{COMMAND}/<CMD>/g' {} \;`

#### NOTE
I switched to shellscripts (**man2txt.sh**, **man2ansi.sh** and **builtin2txt.sh** to get the man pages into utf-8 and remove the control characters with mandoc.

### man2txt.sh

``` bash
#!/bin/bash

while IFS= read -r cmd; do
   mandoc -mdoc -c -T utf8 "$(man -w ${cmd})" | col -b > "$cmd".txt 2>error.err 
done < <(compgen -c | sort | uniq)
find . -type f -empty -delete
```

Mandoc had an issue with outputting {COMMAND} as part of several man pages, which led to import issues where the LLM I used interpretted it as an instruction. To get around this, I switched to using pandoc using man2ansi.sh.

### man2ansi.sh

``` bash
#!/bin/bash

while IFS= read -r cmd; do
  pandoc --from man --to ansi "$(man -w ${cmd})" | col -b >"$cmd".txt 2>error.err
done < <(compgen -c | sort | uniq)
find . -type f -size 1c -delete
```

### builtin2txt.sh
``` bash
#!/bin/bash

while IFS= read -r cmd; do
  echo "$(help ${cmd})" >"$cmd".txt 2>error.err
done < <(compgen -b | sort | uniq)
```

### Architecture

![image](../images/architecture.png)

## Data Ingestion Pipeline for Bashbot

### Imports

In [ ]:
import os
import sys
import shutil
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
)  # ,PyPDFLoader, PyMuPDFLoader for PDFs in the future
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List, Dict, Any, Tuple

### NOTE
If you want to use the existing vector db and skip parsing the docs, set the __USE_EXISTING=True__ below

In [ ]:
USE_EXISTING = False  #  If True, new files will not be added to the vector store.
INIT = True  # If True, te vector store will be deleted and data files will be proessed.
VERBOSE = True  # If True, additional output is generated.
LLM_MODEL = "granite3.3:8b"

### To remove the exiting data store set INIT=True above.
Chromadb is thread-safe but not process safe. The method call in the VectorStore class will generate an error if used within the same process. To avoid this, you can clean the existing vectorstore by manually removing the ../data/vector_store directory or by setting INIT=True above.

## Class Definitions

I want the flexibility to have pluggable implementations. I am going to wrap this part in classes.
Inspiration for this came from Krish Naik's YouTube video
https://youtu.be/o126p1QN_RI?si=7xC5H47A3iuu52RK
and pixegami https://youtu.be/2TJxpyO3ei4?si=zPNEsAFWWy5Cenzq


### Loader

Processes data files and handles the chunking. This notebook is basically a test harness. This code is the canonical representation of loader.py used by the CLI.

In [ ]:
class Loader:
    """This class handles loading of the documents"""

    def __init__(
        self,
        path_name: str = "../data",
        glob: str = "**/*.txt",
        chunk_size=500,
        chunk_overlap=100,
    ):
        """ """
        self.documents = []
        self.path = Path(path_name)
        self.glob = glob
        self.chunks = None
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def _split_documents(self):
        """Chunks the text into chunk_size with an overlap specified"""

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""],
        )
        self.chunks = text_splitter.split_documents(self.documents)
        print(f"Split {len(self.documents)} documents into {len(self.chunks)} chunks.")

    def process_files(self):
        """Processes all files in the path"""

        files = list(self.path.glob(self.glob))
        print(f"Preparing to process {len(files)} text files...")

        for f in files:
            try:
                loader = TextLoader(str(f))
                docs = loader.load()

                for d in docs:
                    d.metadata["manpage"] = Path(f).stem
                    d.metadata["source_file"] = f.name
                    d.metadata["file_type"] = "txt"
                    d.metadata["platform"] = "linux"
                    d.metadata["source"] = "manpage"

                    self.documents.extend(docs)

            except Exception as e:
                print(f"    ERROR {e}")

        print(f"Total documents loaded: {len(self.documents)}.")
        self._split_documents()

### EmbeddingMgr

This code is the canonical representation of embeddings.py used by the CLI.

In [ ]:
import numpy as np
import chromadb
import ollama
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_community.llms import Ollama


class EmbeddingMgr:
    """This class manages the embedding implementation."""

    def __init__(self, embedding_model_name: str = "nomic-embed-text") -> List[Any]:
        """
        ctor

        Args:
            model_name: embedding model name for embeddings.

        Returns:
            List of embeddings
        """

        self.embedding_model_name = embedding_model_name
        self.model = None
        self._load_components()

    def _load_components(self):
        """This method loads the embedding model defined in the ctor."""
        try:
            print(f"Loading Embedding model {self.embedding_model_name}.")
            self.model = OllamaEmbeddings(model=self.embedding_model_name)
            print(f"Embedding Model {self.model} loaded successfully.")
        except Exception as e:
            print(f"Exception while loading {self.embedding_model_name}: {e}")
            raise

    def add_embeddings(self, texts):
        """
        Generate embeddings for a list of text strings

        Args:
            texts: List of strings
        """
        if not self.model:
            raise ValueError("Model has not been loaded.")

        print(f"Generating embeddings for {len(texts)} strings.")

        embeddings = []
        for chunk in texts:
            # generate embeddings
            embedding = ollama.embeddings(
                model=self.embedding_model_name, prompt=chunk.page_content
            )["embedding"]

            embeddings.append(embedding)

        return embeddings

### VectorStore

This code is the canonical representation of vectorstore.py used by the CLI.

In [ ]:
import chromadb
import uuid
from chromadb.config import Settings
from chromadb.utils.batch_utils import create_batches


class VectorStore:
    """Manages embeddings in a vector database"""

    def __init__(
        self,
        collection_name: str = "bashdocs",
        persist_directory: str = "../data/vector_store",
    ):
        """
        ctor

        Args:
            collection_name: Name of the collection
            persist_directory: Path to directory for persistant vectors.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # Create the persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Bash commands document embeddings for RAG"},
            )
            print(f"Vector store initialized for collection {self.collection_name}")
            print(f"Exisiting documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents to the vector store

        Args:
            documents: List of langchain Documents
            embeddings: The embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents to embeddings mismatch.")

        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for idx, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate the uuid
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{idx}"
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = idx
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document Page Content
            documents_text.append(doc.page_content)

            # Embedding vetor
            embeddings_list.append(embedding)

        # Add it to the collection
        try:
            max_batch = 5000  # batch max is 5461
            for i in range(0, len(embeddings), max_batch):
                self.collection.add(
                    embeddings=embeddings_list[i : i + max_batch],
                    documents=documents_text[i : i + max_batch],
                    ids=ids[i : i + max_batch],
                    metadatas=metadatas[i : i + max_batch],
                )
            print(f"Successfully added {len(documents)} documents to the store.")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e})")
            raise

## Notebook Exploration
Determine operting mode for the vector store and remove the store if it is being initalized. This is similar to running the CLI with the -d switch.

In [ ]:
if INIT and not USE_EXISTING:
    store_path = Path("../data/vector_store/")
    print(f"Removing vector store at {store_path}.")
    shutil.rmtree(store_path, ignore_errors=True)

### Ollama Version
This project was prototyped with Grok using a trial API key. The original vision was a local RAG implementation and it has been converted to use Ollama models locally.

**Initialize the Embedding Manager**

Controls the generation of embedding vectors using the model passed in the ctor.

In [ ]:
emb_mgr = EmbeddingMgr("nomic-embed-text")
emb_mgr

**Initialize the Vector Store**

Currently only supports ChromaDB.

In [ ]:
vector_store = VectorStore()
vector_store

In [ ]:
# import ollama
# import chromadb
# from langchain_text_splitters import RecursiveCharacterTextSplitter

### Create and Add embeddings
if USE_EXISTING = False then we read in the files from the data directory and add them to the vector store. This will cause duplicates if run multiple times without setting INIT=True to clean the db store.

In [ ]:
# get the text from the documents and create embeddings to store
if not USE_EXISTING:
    loader = Loader()
    loader.process_files()
    if VERBOSE:
        print(loader.chunks[3])

    texts = [doc for doc in loader.chunks]
    print(type(texts[0]))
    # Use the EmbeddingMgr class to generate them
    embeddings = emb_mgr.add_embeddings(texts)

    # Add the documents to the vector store
    vector_store.add_documents(loader.chunks, embeddings)

## Prompt and RAG Retrieval Pipeline
Support functions to produce queries for testing and evaluation.

In [ ]:
def get_relavent_docs(query, embedding_manager, vector_store, top_k=3):
    # Get embedding for question
    question_embedding = ollama.embeddings(
        model=embedding_manager.embedding_model_name, prompt=query
    )["embedding"]

    # Find relevant chunks
    results = vector_store.collection.query(
        query_embeddings=[question_embedding], n_results=top_k
    )
    #    print(results)
    return results

In [ ]:
def send_question(
    query, llm, embedding_manager, vector_store, top_k=3, pure: bool = False
) -> Tuple:
    context = ""
    found = 0

    if not pure:
        results = get_relavent_docs(query, embedding_manager, vector_store, top_k)
        found = results["ids"]
        if len(found[0]) > 0:
            # Build context from retrieved chunks
            context = "\n\n".join(results["documents"][0])
        else:
            return ("No relavent documents found.", 0)

        prompt = f"""You are a professional AI assistant that specializes in answering
            questions about Linux commands.
            Use ONLY the following context to answer the question accurately and concisely.
            
            Context:
            {context}
            
            Question: {query}
            
            Answer:"""
    else:
        prompt = f"""You are a professional AI assistant that specializes in answering
            questions about Linux commands.
            
            Context:
            {context}
            
            Question: {query}
            
            Answer:"""
    # generate answer
    response = ollama.chat(model=llm, messages=[{"role": "user", "content": prompt}])

    return (found, response["message"]["content"])

In [ ]:
found, result = send_question(
    "how do you delete files?", LLM_MODEL, emb_mgr, vector_store
)
print(f"Found {len(found[0])} documents:\n{found}\n{result}")

## Testing
Read the JSON questions and answers from the test_questions.json (converted from the UTVerse generated data).

Test each question from the JSON file of the top Linux questions generated by UTVerse and capture the LLM answer. Then compare the LLM answer to the answer provided by the chatbot. Looking at the cosine similarity may provide a general evaluation, but a manual inspection will probably be needed for accurate assessment.

### submit a single question and return the answer

In [ ]:
import json

# from langchain_groq import ChatGroq
from langchain_ollama import OllamaLLM
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

### Load the JSON test data

In [ ]:
with open("../test/test_questions.json", "r") as jsf:
    test_data = json.load(jsf)

In [ ]:
type(test_data)

In [ ]:
import pandas as pd

df_test = pd.DataFrame(test_data)

In [ ]:
df_test.head()

In [ ]:
results = []
for item in test_data["linux_questions"][:20]:
    #    result = rag_call(item['question'], fido, llm)
    # send_question(query, llm, embedding_manager, vector_store, top_k=3, pure: bool = False)
    found, result = send_question(item["question"], LLM_MODEL, emb_mgr, vector_store)
    item["result"] = result
    results.append(item)

In [ ]:
len(results)

In [ ]:
df_results = pd.DataFrame(results)

In [ ]:
df_results.head()

In [ ]:
df_results["result"][0]

## Function Definitions

#### Helper to calculate cosine similarity between the provided JSON answer and the LLM answer

In [ ]:
def gen_embeddings(text, embedding_manager) -> str:
    # Generate embeddings for a given str
    return ollama.embeddings(model=embedding_manager.embedding_model_name, prompt=text)[
        "embedding"
    ]

In [ ]:
import numpy as np
from numpy.linalg import norm


def calculate_similarity(x, y):
    """Calculates the cosine similarity between 2 embeddings"""
    cosine = np.dot(x, y) / (norm(x) * norm(y))
    return cosine

#### Test Loop

In [ ]:
def test_similarity(
    data: Dict = None, iterations: int = 20, pure: bool = False, top_k: int = 3
) -> List[Dict]:
    results = []

    for item in data["linux_questions"][:iterations]:
        found, result = send_question(
            item["question"], LLM_MODEL, emb_mgr, vector_store, pure=pure
        )
        X = gen_embeddings(item["answer"], emb_mgr)
        Y = gen_embeddings(result, emb_mgr)
        similarity = calculate_similarity(X, Y)
        item["result"] = result
        item["similarity"] = similarity
        #        item['context'] = result[2]
        results.append(item)

    return results

In [ ]:
results = test_similarity(test_data, pure=False)

#### Average Similarity (with RAG)

In [ ]:
df = pd.DataFrame(results)
df.head()
print(df["similarity"].mean())

In [ ]:
df["similarity"].describe()

In [ ]:
df["similarity"].value_counts()

### Test without RAG

In [ ]:
results = test_similarity(test_data, pure=True)

#### Average Similarity (without RAG)

In [ ]:
df = pd.DataFrame(results)
df.head()
print(df["similarity"].mean())

In [ ]:
df["similarity"].describe()

In [ ]:
df["similarity"].value_counts()

### Test with RAG and _top_k=2_

In [ ]:
results = test_similarity(test_data, top_k=2, iterations=20, pure=False)

In [ ]:
df = pd.DataFrame(results)
df.head()
print(df["similarity"].mean())

In [ ]:
df["similarity"].describe()

In [ ]:
df["similarity"].value_counts()

In [ ]:
results = test_similarity(
    test_data, iterations=len(test_data["linux_questions"]), pure=False
)

In [ ]:
len(results)

In [ ]:
df = pd.DataFrame(results)
df.head()
print(df["similarity"].mean())

In [ ]:
df["similarity"].describe()

In [ ]:
df["similarity"].value_counts()

## Conclusions
Choosing models is important for RAG performance. I was able to get comparable results between Grok and the llama3 local LLM. The processing of the input documents was performed locally , however chunking allowed the GPU utilization to be relatively steady at 12%-30% on my Nvidia 4070.

![Processing Chunks](../images/screenshot_btop_nomic.png "Nomic model")

The real GPU utilization comes from the LLM model processing a query.

![llam3 processing](../images/screenshot-btop_llama3.png "llama3 model")

Note the GPU spike as well as the jump in wattage required from 42.7 for chunk processing to 199 for LLM processing.

## Future Direction


I began with Grok since it is easy to generate an API key, however my main goal is to use a local Ollama model and only return results from the vector_store. I have installed Ollama and am running it as a systemd servie on both Debian 13 (Trixie). My end goal is to create a docker container that can run efficiently on CPU in an OCI container. This would allow me to deploy a container to a server and allow users to import additional documents into the existing store to share with the development team.

To support multiple document formats, I intened to integrate Docling into the input file processing pipeline. This will enable robust support for supported formats without having to juggle a bunch of implementations to import word docs, pdfs, etc...

Once that is completed, I will create a web UI using OpenWebUI to create the UX and allow the specification of personality, as well as the system prompts to enable finer control over output from various users. Eventually users will be able to upload documents to add to the store on the host machine to provide a secure central LLM focused on the source code, documents and database artifacts for a single user or specific team. The data that matetrs in my daily work involves data involving these artifacts and I hope to have a small focused LLM that is more accurate at generating meaningful answers to my questions during the design of new project architecture.

I rebuilt my dev setup to be able to work on my laptop, but ssh into my desktop system to run the models and utilize my GPU. For future versions I will be converting the projct to a literate version using Emacs orgmode and TRAMP to have the documentation and the source code in a single text file with control to spin up containers to run Ollama as well as Jupyter kernels on the GPU host to allow remote exploratory work to be continues on a "live" system. To complete the full vision will require a new Lisp package I am working on to supplement ein and tramp to allow control over the entire dev pipeline from my Emacs environment.

### References

IBM Technology
LangChain vs LangGraph: A Tale of Two Frameworks
https://youtu.be/qAF1NjEVHhY?si=CKShLFYuOz0xbi1N

Krish Naik
Complete RAG Crash Course With Langchain In 2 Hours
https://youtu.be/o126p1QN_RI?si=7xC5H47A3iuu52RK

pixegami
Python RAG Tutorial (with Local LLMs): AI For Your PDFs
https://youtu.be/2TJxpyO3ei4?si=zPNEsAFWWy5Cenzq

UTVerse
https://utverse.tennessee.edu/

Technologies, Cuantum. Natural Language Processing with Python Updated Edition: From Basics to Advanced Projects: Mastering Text Analysis, Machine Learning Models, and Chatbot ... (Mastering the AI Revolution Book 4) (p. 1). (Function). Kindle Edition. 

Tunstall, Lewis; Werra, Leandro von; Wolf, Thomas. Natural Language Processing with Transformers, Revised Edition (p. 4). (Function). Kindle Edition. 

“The Complete Guide to Ollama: Run Large Language Models Locally.” DEV Community, 16 Feb. 1970, dev.to/ajitkumar/the-complete-guide-to-ollama-run-large-language-models-locally-2mge. Accessed 28 Mar. 2026. 